In [ ]:
from urllib.request import Request, urlopen 
from bs4 import BeautifulSoup

def remove_content_between_chars(string: str, start="[", end="]"):
    edited = []
    skip = False
    for char in string:
        if char == start:
            skip = True

        if not skip:
            edited.append(char)

        if char == end:
            assert skip, f"Closing character found without opening in {''.join(edited)}"
            skip = False
    return "".join(edited)


In [ ]:
import csv
import json
import re
from typing import List, Set

from spacy.tokens.doc import Doc
from spacy.tokens import Span
from tqdm import tqdm
from unidecode import unidecode

MODEL_TYPE = "visual" #visual, detection, text-only

def get_paragraphs(url: str):
        headers = {
        "User-Agent": "Mozilla/5.0 (compatible; WikipediaScraper/1.0; +https://example.com/contact)"
        }
        req = Request(url, headers=headers)
        page = urlopen(req)

        assert page.getcode() == 200, f"{url} is not valid"
        soup = BeautifulSoup(page, "html.parser")

        res = [soup.find("h1")]
        div = soup.find("div", class_="mw-content-ltr mw-parser-output")
        res += [i for i in div.find_all(re.compile("(h.)|p"))]

        paragraphs = []
        for tag in res:
            if tag.name.startswith("h"):
                current_header = tag
            elif tag.name == "p" and tag.get("style") is None:
                # remove multiple whitespaces
                p = " ".join(tag.text.split())
                # remove additional whitespaces
                p = p.strip()
                # remove content between '[' and ']'
                p = remove_content_between_chars(p)
                if len(p) > 0:
                    paragraphs.append({
                        "text": p,
                        "header_name": current_header.text,
                        "header_level": current_header.name[-1]
                    })
        
        return paragraphs
        
        
def check_exact_match(paragraph: str, answer: str):
    # case insensitive match
    return bool(re.search(f"(^|[^\w]{{1}}){answer}($|[^\w]{{1}})", paragraph, flags=re.IGNORECASE))


def check_full_match(paragraph: str, attribute: str, answer: str):
    # a full match requires to match both attribute (e.g. President) and answers
    for to_find in [attribute, answer]:
        if not check_exact_match(paragraph, to_find):
            return False
    return True


def write_roman(num: int):
    ROMAN = {
        1000: "M",
        900: "CM",
        500: "D",
        400: "CD",
        100: "C",
        90: "XC",
        50: "L",
        40: "XL",
        10: "X",
        9: "IX",
        5: "V",
        4: "IV",
        1: "I",
    }

    def roman_num(num: int):
        for r in ROMAN.keys():
            x, y = divmod(num, r)
            yield ROMAN[r] * x
            num -= r * x
            if num <= 0:
                break

    return "".join([a for a in roman_num(num)])


def remove_additional_bits(string: str, additional_bits: List[str]):
    for bit in additional_bits:
        string = re.sub(bit, "", string)
    return " ".join(string.split())  # remove additional whitespaces


def find_main_chunk(doc: Doc):
    ancestor = None
    for chunk in doc.noun_chunks:
        if ancestor is None:
            ancestor = chunk
        elif chunk.root.is_ancestor(ancestor.root):
            ancestor = chunk.root
    return ancestor


def is_monarch(span: Span, monarch_nums: Set[str]):
    for name_chunk in span.text.split():
        if name_chunk in monarch_nums:
            return True
    return False

def refine_answer(answer: str):
    # Clean the answer from dates and times
    return answer.split("|")[0].strip()

In [ ]:
import os
import spacy
from spacy.tokenizer import Tokenizer

MONARCH_NUMS = {write_roman(i) for i in range(1, 100, 1)}

ADDITIONAL_BITS = [
    "[\w]?F(\.)?C(\.)?[\w]?",  # FC, F.C., AFC, ... for Football
    "[\w]?C(\.)?F(\.)?[\w]?",  # CF, ... for Football
    "[\w]?F(\.)?K(\.)?[\w]?",  # FK, ... for Football
    "[\w]?A(\.)?S(\.)?[\w]?",  # AS, ... for Football
    "[\w]?S(\.)?V(\.)?[\w]?",  # SV, ... for Football
    "[\w]?B(\.)?C(\.)?[\w]?",  # BC, ... for Basketball
    "[\w](\.)[\w](\.)",  # General regex for to remove two letter acronyms (with )
    "football",
    "(t|T)eam",
    "association",
    "men's",
    "basketball",
    "F1",
    "(S|s)cuderia",
    "(R|r)acing",
]

EXCEPTIONS_ANSWERS = {
    "Kōji Seto": "Koji Sato", 
    "Salman bin Abdulaziz Al Saud": "Salman" #The head noun is selected as Saud wrt Salman; Salman in wikidata is both king and prime minister while in wikipedia no
    } 

nlp = spacy.load("en_core_web_trf")
nlp.tokenizer = Tokenizer(nlp.vocab)  # Whitespace tokenization

def json_load(path: str):
    with open(path, "r") as f:
        return json.load(f)

def extract_passages(dolma_selected_documents: dict, sampled_entities: dict, nlp: spacy.Language) -> dict:
    """
    Extract passages from dolma_selected_documents for each item in sampled_entities.
    """
    passages = {}

    for category, items in tqdm(sampled_entities.items(), total=len(sampled_entities)):
        for item, attributes in items.items():
            for attribute, data in attributes.items():

                answers = data["answers"]

                paragraphs = []
                if item in dolma_selected_documents:
                    page = dolma_selected_documents[item]
                    paragraphs = page["text"].split("\n")
                elif item == "Cencora, Inc.":
                    page = dolma_selected_documents["Cencora"]
                    paragraphs = page["text"].split("\n")
                else:
                    raise ValueError(f"No page found for {item} in dolma selected documents")

                attribute = attribute if attribute != "Sports Team" else None
                
                if category not in passages:
                    passages[category] = {}
                
                if item not in passages[category]:
                    passages[category][item] = {}

                if attribute is not None:
                    if attribute not in passages[category][item]:
                        passages[category][item][attribute] = {}

                matches = {
                    "full": [],
                    "em": [],
                    "simplified": [],
                    "head": [],
                }

                no_match = True
                for answer in tqdm(answers, desc=f"answers for {item}"):
                    original_answer = answer

                    for paragraph in paragraphs:
                        answer = refine_answer(original_answer)

                        if answer in EXCEPTIONS_ANSWERS:
                            answer = EXCEPTIONS_ANSWERS[answer]
                        append_to = None
                        matched = None

                        attr = attribute
                        if attr is not None:
                            attr_l = attr.lower()

                            if "prime minister" in attr_l:
                                attr = ["prime minister"]
                            elif "president" in attr_l:
                                attr = ["president"]
                            elif "chancellor" in attr_l:
                                attr = ["chancellor"]
                            elif "king" in attr_l:
                                attr = ["king"]
                            elif "emperor" in attr_l:
                                attr = ["emperor"]
                            elif "monarch" in attr_l:
                                attr = ["monarch"]
                            elif "supreme leader" in attr_l:
                                attr = ["supreme leader"]
                            elif "premier" in attr_l:
                                attr = ["premier"]
                            elif "chairperson" in attr_l:
                                attr = ["chairperson", "chairman", "chairwoman"]
                            elif "chief executive officer" in attr_l:
                                attr = ["chief executive officer", "ceo"]
                            else:
                                attr = [attr]
                        else:
                            attr = [None]

                        # ---------- 1. FULL (attribute + answer) ----------
                        for a in attr:
                            if a is not None and check_full_match(paragraph, a, answer):
                                append_to = "full"
                                matched = (a, answer)
                                no_match = False
                                break

                        # ---------- 2. EM (exact answer only) ----------
                        if append_to is None and check_exact_match(paragraph, answer):
                            append_to = "em"
                            matched = answer
                            no_match = False

                        # ---------- 3. SIMPLIFIED ----------
                        if append_to is None:

                            if category in ["athletes"]:
                                answer = remove_additional_bits(answer, ADDITIONAL_BITS)

                            if check_exact_match(paragraph, answer):
                                append_to = "simplified"
                                matched = answer
                                no_match = False

                        # ---------- 4. HEAD ----------
                        if append_to is None and len(answer.split()) > 1:
                            doc = nlp(answer)
                            main_chunk = find_main_chunk(doc)

                            if main_chunk is not None:
                                if is_monarch(main_chunk, MONARCH_NUMS):
                                    answer = main_chunk.text
                                else:
                                    answer = main_chunk.root.text

                                if check_exact_match(paragraph, answer):
                                    append_to = "head"
                                    matched = answer
                                    no_match = False

                        if append_to:
                            matches[append_to].append({
                                "answer": original_answer,
                                "paragraph": paragraph,
                                "matched": matched
                            })
                if attribute is not None:
                    if attribute not in passages[category][item]:
                        passages[category][item][attribute] = {}

                    passages[category][item][attribute] = {
                        "matches": matches,
                        "no_match": no_match
                    }
                else:
                    passages[category][item] = {
                        "matches": matches,
                        "no_match": no_match
                    }
    return passages

dolma_selected_documents = json_load(f"data/dolma_selected_documents/{MODEL_TYPE}/data.json")

for elem in ["correct", "outdated", "irrelevant"]:
    sampled_entities = json_load(f"data/sampled_entities/{MODEL_TYPE}/{elem}/wikidata.json")
    passages = extract_passages(dolma_selected_documents, sampled_entities, nlp)

    os.makedirs(f"data/passages/{MODEL_TYPE}/{elem}/", exist_ok=True)


    with open(f"data/passages/{MODEL_TYPE}/{elem}/passages.json", "w") as f:
        json.dump(passages, f, indent=4)
